# PromptWar — Training Notebook

This notebook is a one-stop driver for everything `training/train.py` can do, plus the env server it talks to. It mirrors the four CLI modes:

| `--mode`  | What it runs                                                       | Needs                                |
|-----------|--------------------------------------------------------------------|--------------------------------------|
| `smoke`   | Stub env + scripted policy (5 GRPO steps; skips GRPO if no torch)  | CPU only                             |
| `live`    | Real env URL + scripted policy, 1 end-to-end episode               | Env server                           |
| `baseline`| Random policy, N episodes; per-role reward aggregates              | Env server (or stub via `--env-url stub`) |
| `long`    | Real env + Qwen2.5-3B base + 3 LoRA adapters + 3 GRPOTrainers      | GPU, torch/trl/peft, ideally unsloth + 4-bit |

Run cells top-to-bottom. CPU-only Colab is enough for **smoke**, **live**, and **baseline** (rubric fallback). The **long** GRPO run needs a T4/L4/A100 plus the Consumer Model loaded for meaningful A/S rewards.

## 1. Clone the repo

On Colab, this drops the project under `/content/meta-hackathon`. Locally, just skip this cell — it's a no-op when the directory already exists.

In [ ]:
%cd /content
![ -d meta-hackathon ] || git clone https://github.com/rishabhshukla0912/meta-hackathon.git
%cd /content/meta-hackathon
!git pull --ff-only || true
!ls

## 2. Install dependencies

Three layers — install only what your runtime needs:

* **Layer 1** is always required (env server + HTTP client + tokenizer).
* **Layer 2** loads the Qwen2.5-0.5B Consumer Model used by the rubrics. Without it, A and S rewards fall back to deterministic mocks.
* **Layer 3** is only needed for `--mode long` (real GRPO training).

In [ ]:
%pip install -q "openenv-core @ git+https://github.com/meta-pytorch/OpenEnv.git"
%pip install -q fastapi "uvicorn[standard]" httpx pydantic regex "tokenizers>=0.22" "transformers>=4.56,<6"

In [ ]:
%pip install -q "accelerate>=1.0"
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
%pip install -q "peft>=0.13" "trl>=0.11,<0.16" "datasets>=3.0" "bitsandbytes>=0.43"

## 3. Verify install with the trainer-side test suite

Ten small unit tests under `training/tests/`. They don't need torch/trl/peft — they exercise the rollout router, scripted policy, and trainer wiring.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m unittest discover -s training/tests -v

## 4. Start the env server

Boots `PromptWar_env.server.app:app` on `127.0.0.1:8000`. The Consumer Model is **not** loaded eagerly — see the next cell to load it on demand.

In [ ]:
import os, subprocess, time, signal, pathlib, httpx

ROOT = pathlib.Path('/content/meta-hackathon')
if not ROOT.exists():
    ROOT = pathlib.Path('.').resolve()
LOG  = ROOT / 'server.log'
PIDFILE = ROOT / 'server.pid'

# Kill any previous instance from this notebook session.
if PIDFILE.exists():
    try:
        os.kill(int(PIDFILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except ProcessLookupError:
        pass
    PIDFILE.unlink(missing_ok=True)

env_vars = os.environ.copy()
env_vars['PYTHONPATH'] = str(ROOT)
# Set to '1' to load Qwen2.5-0.5B-Instruct eagerly at startup (GPU + ~1.5 GB VRAM).
env_vars['PROMPTWAR_LOAD_CONSUMER_MODEL'] = '0'

with open(LOG, 'wb') as logf:
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'PromptWar_env.server.app:app',
         '--host', '127.0.0.1', '--port', '8000'],
        cwd=str(ROOT), env=env_vars, stdout=logf, stderr=subprocess.STDOUT,
    )
PIDFILE.write_text(str(proc.pid))
print(f'started uvicorn  pid={proc.pid}  log={LOG}')

time.sleep(3)
ok = False
for i in range(30):
    if proc.poll() is not None:
        print(f'uvicorn exited with code {proc.returncode} — see log')
        break
    try:
        r = httpx.get('http://127.0.0.1:8000/health', timeout=5.0)
        if r.status_code == 200:
            print(f'server is up (attempt {i+1}):  {r.json()}')
            ok = True
            break
    except Exception as exc:
        if i % 5 == 0:
            print(f'  attempt {i+1}: {type(exc).__name__}: {exc}')
    time.sleep(1)

if not ok:
    print('server did not come up — check the log below')

print('--- last 30 lines of server.log ---')
!tail -n 30 {LOG}

## 5. (Optional) Load the Consumer Model

Without it, the A/S rubrics fall back to deterministic keyword heuristics — fine for `smoke` / `live` / `baseline`, but the `long` GRPO run won't learn anything for A and S. First call downloads ~1.5 GB.

In [ ]:
import httpx

status = httpx.get('http://127.0.0.1:8000/consumer/status', timeout=10.0).json()
print('before load:', status)

if not status.get('available'):
    print('loading Consumer Model — this can take ~60 s on first run...')
    load = httpx.post('http://127.0.0.1:8000/consumer/load', timeout=600.0).json()
    print('load result:', load)

status = httpx.get('http://127.0.0.1:8000/consumer/status', timeout=10.0).json()
print('after load: ', status)
if not status.get('available'):
    print('\n  !!  Consumer Model unavailable — rubrics will use deterministic mocks.')
    print('  !!  Real GRPO training (--mode long) will NOT produce meaningful rewards for A and S.')
else:
    print('\n  OK  rubrics will use the live Qwen2.5-0.5B-Instruct model.')

## 6. `--mode smoke` — trainer-wiring sanity check

Drives the **stub env** with the scripted policy for 5 episodes. Verifies `role_router` produces sane per-role transition lists. If torch/trl are installed it also takes one real GRPO step; if not, it logs a warning and continues. **No GPU and no live env required.**

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode smoke --steps 5

## 7. `--mode live` — one episode end-to-end

Hits the real env URL with the scripted policy for one episode. Confirms the rollout shape, env wire format, and reward routing all line up between trainer and env.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode live --env-url http://127.0.0.1:8000 --episodes 1

## 8. `--mode baseline` — random-policy reference numbers

30 episodes with a uniform-random edit policy. The aggregate mean per episode answers the §19 question "is the env trivially solvable by random play?". Saves the run output to `training/logs/baseline_<ts>.log`.

In [ ]:
%cd /content/meta-hackathon
import datetime, pathlib
pathlib.Path('training/logs').mkdir(parents=True, exist_ok=True)
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = f'training/logs/baseline_{ts}.log'
print(f'log: {log_path}')
!PYTHONPATH=. python3 -u -m training.train --mode baseline \
    --env-url http://127.0.0.1:8000 \
    --episodes 30 --seed 0 \
    --log-level WARNING 2>&1 | tee {log_path}

## 9. Pre-flight before the long GRPO run

Two checks decide whether `--mode long` will produce any learning signal:

1. **Consumer Model loaded?** — drives the A and S rubrics. If not loaded, both fall back to keyword heuristics and stay constant.
2. **Which GRPO step path is available?** — TRL's native rollout API if the installed version exposes it, otherwise the in-repo whitened-advantage REINFORCE with KL to the LoRA-disabled base.

In [ ]:
import httpx
cm = httpx.get('http://127.0.0.1:8000/consumer/status', timeout=10.0).json()
cm_ok = bool(cm.get('available'))
print(f'[1] Consumer Model: {"LOADED" if cm_ok else "NOT LOADED"} | model_id={cm.get("model_id")} | err={cm.get("load_error")!r}')
if not cm_ok:
    print('    → A and S rewards will be deterministic mocks. Re-run the consumer-load cell first.')

print('\n[2] GRPO step path:')
try:
    from trl import GRPOTrainer
    rollout_api_methods = [m for m in (
        'training_step_with_rollouts',
        '_generate_and_score_completions',
        'compute_rewards',
        '_inner_training_loop',
    ) if hasattr(GRPOTrainer, m)]
    print('    GRPOTrainer methods present:', rollout_api_methods or '(none)')
    if 'training_step_with_rollouts' in rollout_api_methods:
        print('    → using TRL native rollout API.')
    else:
        print('    → using in-repo _grpo_step (whitened advantage + KL).')
except ImportError as e:
    print(f'    trl not installed ({e}) — long mode will fail at trainer construction.')

print('\nIf [1] is LOADED you should see per-role reward variation in the long run below.')

## 10. `--mode long` — full GRPO training (GPU)

Loads Qwen2.5-3B-Instruct + three LoRA adapters (one per role) and runs `--steps` iterations of GRPO. Writes `metrics_log.json` after every step (crash-safe) and checkpoints adapters every `--checkpoint-every` steps under `--output-dir`.

Start with 50 steps as a burn-in. Once pre-flight is green and per-role rewards vary, bump to 200–1500.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode long \
    --env-url http://127.0.0.1:8000 \
    --steps 50 \
    --load-in-4bit \
    --checkpoint-every 25 \
    --output-dir ./checkpoints/promptwar \
    --log-level INFO 2>&1 | tee /content/meta-hackathon/long_run.log | tail -n 200

## 11. Quick post-run diagnostic

Walks the per-step records in `long_run.log` and reports whether each role's `mean_reward` is varying or stuck — fast "is anything learning?" check before plotting curves.

In [ ]:
import re, pathlib

log = pathlib.Path('/content/meta-hackathon/long_run.log')
if not log.exists():
    log = pathlib.Path('long_run.log')
txt = log.read_text() if log.exists() else ''
if not txt:
    print('No long_run.log found — run the long-mode cell first.')
else:
    series = {'A': [], 'S': [], 'B': []}
    for step_dict in re.finditer(r"\{'step': (\d+), 'metrics': \[(.*?)\]\}(?=, \{'step|\])", txt):
        for role_block in re.finditer(r"'role': '([ASB])'.*?'mean_reward': ([-\d.]+)", step_dict.group(2)):
            series[role_block.group(1)].append(float(role_block.group(2)))
    for role in 'ASB':
        rewards = series[role]
        if not rewards:
            continue
        unique_vals = len(set(round(r, 3) for r in rewards))
        verdict = 'CONSTANT (mock-rubric trap?)' if unique_vals == 1 else 'varying'
        print(f'  {role}: {len(rewards)} steps | first={rewards[0]:.2f}  last={rewards[-1]:.2f}  '
              f'min={min(rewards):.2f}  max={max(rewards):.2f}  unique={unique_vals}  → {verdict}')

## 12. Plot training curves

Reads `checkpoints/promptwar/metrics_log.json` (written every step by `run_long`) and plots per-role `mean_reward`, `pg_loss`, and `KL(π‖π_ref)` with a 5-step rolling mean. Safe to run mid-training.

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt

candidates = [
    pathlib.Path('/content/meta-hackathon/checkpoints/promptwar/metrics_log.json'),
    pathlib.Path('checkpoints/promptwar/metrics_log.json'),
]
LOG_PATH = next((p for p in candidates if p.exists()), None)

if LOG_PATH is None:
    print('no metrics file found — run the long-mode training cell first')
else:
    raw = json.loads(LOG_PATH.read_text())
    print(f'loaded {len(raw)} steps from {LOG_PATH}')

    series = {'A': [], 'S': [], 'B': []}
    for entry in raw:
        step_idx = entry['step']
        for role_block in entry['metrics']:
            role = role_block.get('role')
            inner = role_block.get('metrics') or {}
            if role not in series or not inner:
                continue
            series[role].append({
                'step': step_idx,
                'mean_reward': inner.get('mean_reward'),
                'pg_loss': inner.get('pg_loss'),
                'kl': inner.get('kl'),
            })

    def smooth(xs, k=5):
        out = []
        for i in range(len(xs)):
            window = [v for v in xs[max(0, i - k + 1): i + 1] if v is not None]
            out.append(sum(window) / len(window) if window else None)
        return out

    role_color = {'A': 'tab:blue', 'S': 'tab:orange', 'B': 'tab:green'}
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for role, rows in series.items():
        if not rows:
            continue
        steps = [r['step'] for r in rows]
        reward = [r['mean_reward'] for r in rows]
        pg = [r['pg_loss'] for r in rows]
        kl = [r['kl'] for r in rows]
        c = role_color[role]
        for ax, ys in zip(axes, (reward, pg, kl)):
            ax.plot(steps, ys, alpha=0.25, color=c)
            ax.plot(steps, smooth(ys), color=c, linewidth=2.0, label=f'agent {role}')
    titles = ['mean_reward per role', 'pg_loss (-A · log π)', 'KL(π ‖ π_ref)']
    for ax, title in zip(axes, titles):
        ax.set_title(title); ax.set_xlabel('step'); ax.grid(True, alpha=0.3); ax.legend(loc='best', fontsize=9)
    axes[0].set_ylabel('reward'); axes[1].set_ylabel('pg_loss'); axes[2].set_ylabel('kl')
    fig.suptitle('PromptWar GRPO training curves (raw + 5-step rolling mean)', fontsize=12)
    fig.tight_layout()
    plt.show()

## 13. Stop the env server

Cleans up the uvicorn process started in cell 4.

In [ ]:
import os, signal, pathlib
for candidate in (pathlib.Path('/content/meta-hackathon/server.pid'), pathlib.Path('server.pid')):
    if candidate.exists():
        pid = int(candidate.read_text().strip())
        try:
            os.kill(pid, signal.SIGTERM)
            print(f'stopped uvicorn pid={pid}')
        except ProcessLookupError:
            print(f'pid {pid} already gone')
        candidate.unlink(missing_ok=True)
        break
else:
    print('no server.pid — nothing to stop')

### Tips

* **CPU-only Colab**: cells 6 (smoke), 7 (live), 8 (baseline) all run fine. Skip the Consumer-Model cell and skip cell 10 (`--mode long`). The deterministic rubric fallbacks still produce non-trivial baseline numbers (~27/45 on a random policy).
* **GPU Colab (T4/L4)**: load the Consumer Model (cell 5), then add `--use-unsloth` to the `--mode long` command for ~2× throughput. With `--load-in-4bit` you can fit Qwen2.5-3B + 3 LoRA adapters in ~12 GB VRAM.
* **Long overnight run**: bump `--steps 50` → `--steps 1500` and `--checkpoint-every 25` → `--checkpoint-every 100`. The metrics log is rewritten every step, so the curve-plot cell is safe to run mid-training to peek at progress.